# Aegis-Safe-Work — Fall Detection ETL · Stage 3

**Tensorization.** Turns each Stage-1 clip into a model-ready tensor: sample **16 frames**, ImageNet-normalize, save as `float16` `.npy` under `tensors/{split}/{class}/`, and emit a flat `manifest.csv` for the dataloader.

- Reads the Stage-2 `split_manifest.json`; output tensor shape per clip: **(16, 3, 224, 224)**.
- Final artifact: **2085 tensors** + `manifest.csv` (`tensor_path, label, split`) — the direct input to training.

In [ ]:
!pip install opencv-python-headless numpy --quiet

In [ ]:
import csv
import json
import time
from datetime import datetime, timezone
from pathlib import Path

In [ ]:
import cv2
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DRIVE_ROOT   = Path("/content/drive/MyDrive")
PROJECT_ROOT = DRIVE_ROOT / "Aegis_Safe_Work"

In [ ]:
STAGE1_ROOT  = PROJECT_ROOT / "processed/stage1"
TENSORS_ROOT = PROJECT_ROOT / "tensors"
MANIFEST_DIR = PROJECT_ROOT / "manifests"
SPLIT_JSON   = MANIFEST_DIR / "split_manifest.json"
MANIFEST_CSV = MANIFEST_DIR / "manifest.csv"

## Config — clip shape, normalization & label maps

**16 frames** at **224x224**, ImageNet mean/std, class-dir map (`Fall`->`fall`), and integer labels (**Fall=1, Normal=0**). Paths point at Stage-1 videos in, `tensors/` + `manifest.csv` out.

In [ ]:
N_FRAMES     = 16
IMG_SIZE     = 224

In [ ]:
# ImageNet normalization
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


In [ ]:
# stage1 subdir (lowercase) per manifest key
LABEL_TO_DIR = {
    "Fall":   "fall",
    "Normal": "normal",
}

In [ ]:
# integer label for manifest.csv
LABEL_INT = {
    "Fall":   1,
    "Normal": 0,
}

## Core — frame extraction

`extract_frames` samples **16 frames uniformly** (`linspace` over the clip), converts BGR->RGB, scales to [0,1]. Duplicates the last valid frame on a read miss; returns `None` for unopenable/empty videos.

In [ ]:

def extract_frames(video_path: Path, n_frames: int = N_FRAMES) -> np.ndarray | None:
    """
    Open video, sample n_frames uniformly via linspace over total frame count.
    Returns float32 array (n_frames, H, W, 3) BGR->RGB, pixels in [0,1].
    Returns None if video cannot be opened or has insufficient frames.
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < 1:
        cap.release()
        return None

    # linspace indices, clamped to [0, total-1]
    indices = np.linspace(0, total - 1, n_frames, dtype=int)
    indices = np.clip(indices, 0, total - 1)

    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            # fallback: use last valid frame duplicated
            if frames:
                frames.append(frames[-1].copy())
            else:
                cap.release()
                return None
            continue
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame_rgb)

    cap.release()

    if len(frames) != n_frames:
        return None

    arr = np.stack(frames, axis=0).astype(np.float32) / 255.0  # (16, H, W, 3)
    return arr


## Core — tensor normalization

`to_tensor` applies ImageNet mean/std, transposes to channels-first **(16, 3, 224, 224)**, and casts to **float16** to halve on-disk size. This normalization must match the backend's `normalize_frame` exactly.

In [ ]:

def to_tensor(frames: np.ndarray) -> np.ndarray:
    """
    frames : (16, 224, 224, 3) float32 in [0,1]
    output : (16, 3, 224, 224) float16  ImageNet normalized
    """
    normalized = (frames - MEAN) / STD            # (16, H, W, 3) float32
    transposed = normalized.transpose(0, 3, 1, 2) # (16, 3, H, W) float32
    return transposed.astype(np.float16)



## Per-block processing

`process_block` converts every video of one (split, label) into a `.npy` tensor: **resume-safe** (skips existing), logs progress + ETA every 100 files, records missing/failed sources, and appends manifest rows in place.

In [ ]:

def process_block(
    split: str,
    label_cap: str,
    filenames: list[str],
    records: list[dict],
) -> dict:
    """
    Process all videos for one (split, label) block.
    Appends manifest records to `records` list in place.
    Returns summary dict.
    """
    label_dir = LABEL_TO_DIR[label_cap]
    src_dir   = STAGE1_ROOT / label_dir
    out_dir   = TENSORS_ROOT / split / label_dir
    out_dir.mkdir(parents=True, exist_ok=True)

    label_int = LABEL_INT[label_cap]
    total     = len(filenames)
    ok        = 0
    skipped   = 0
    errors    = 0
    error_files = []
    t_start   = time.time()

    for idx, fname in enumerate(filenames, start=1):
        stem    = Path(fname).stem          # e.g. Fall_0001
        npy_out = out_dir / f"{stem}.npy"

        # Resume support
        if npy_out.exists() and npy_out.stat().st_size > 0:
            skipped += 1
            records.append({
                "tensor_path": str(npy_out),
                "label":       label_int,
                "split":       split,
            })
            if idx % 100 == 0:
                print(f"  [{idx}/{total}] SKIP: {stem}.npy")
            continue

        src = src_dir / fname
        if not src.exists():
            print(f"  [{idx}/{total}] MISSING source: {fname}")
            errors += 1
            error_files.append(str(src))
            continue

        frames = extract_frames(src, N_FRAMES)
        if frames is None:
            print(f"  [{idx}/{total}] ERROR extracting frames: {fname}")
            errors += 1
            error_files.append(str(src))
            continue

        tensor = to_tensor(frames)          # (16, 3, 224, 224) float16
        np.save(str(npy_out), tensor)
        ok += 1

        records.append({
            "tensor_path": str(npy_out),
            "label":       label_int,
            "split":       split,
        })

        if idx % 100 == 0 or idx == total:
            elapsed = time.time() - t_start
            rate    = ok / elapsed if elapsed > 0 else 0
            eta_s   = (total - idx) / rate if rate > 0 else 0
            print(
                f"  [{idx}/{total}] OK: {stem}.npy | "
                f"rate={rate:.1f} t/s | ETA={eta_s/60:.1f} min"
            )

    return {
        "split":      split,
        "label":      label_cap,
        "total":      total,
        "ok":         ok,
        "skipped":    skipped,
        "errors":     errors,
        "error_files": error_files,
        "elapsed_min": (time.time() - t_start) / 60,
    }



## Orchestrator — build tensors + manifest.csv

`main` reads `split_manifest.json` (from Stage 2), runs `process_block` over every (split x class), then writes `manifest.csv` (`tensor_path, label, split`) — the flat index the training dataloader reads. Labels: **Fall=1, Normal=0**.

In [ ]:

def main():
    print("=" * 60)
    print("Aegis-Safe-Work | FallDetection ETL Stage 3")
    print(f"Started: {datetime.now(timezone.utc).isoformat()}")
    print("=" * 60)

    # Load split manifest
    if not SPLIT_JSON.exists():
        raise FileNotFoundError(f"split_manifest.json not found: {SPLIT_JSON}")

    with open(SPLIT_JSON, encoding="utf-8") as f:
        manifest = json.load(f)

    TENSORS_ROOT.mkdir(parents=True, exist_ok=True)
    MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

    records   = []   # manifest.csv rows
    summaries = []

    for split in ["train", "val"]:
        for label_cap in ["Fall", "Normal"]:
            filenames = manifest[split][label_cap]
            print(f"\n[{split.upper()} / {label_cap.upper()}] {len(filenames)} videos")
            summary = process_block(split, label_cap, filenames, records)
            summaries.append(summary)

    # Write manifest.csv
    with open(MANIFEST_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["tensor_path", "label", "split"])
        writer.writeheader()
        writer.writerows(records)

    print(f"\n[MANIFEST CSV] Written: {MANIFEST_CSV} ({len(records)} rows)")

    # Final summary
    print("\n" + "=" * 60)
    print("ETL STAGE 3 SUMMARY")
    print("=" * 60)
    grand_ok = 0
    for s in summaries:
        print(
            f"  {s['split']:<5} / {s['label']:<6} | "
            f"ok={s['ok']:>5} skip={s['skipped']:>5} "
            f"err={s['errors']:>3} | {s['elapsed_min']:.1f} min"
        )
        grand_ok += s["ok"] + s["skipped"]
        if s["error_files"]:
            for ef in s["error_files"]:
                print(f"    ERROR: {ef}")
    print(f"\n  Total tensors in tensors/: {grand_ok}")
    print("=" * 60)
    print("Stage 3 complete. tensors/ ready for training.")


## Run

Result: **2085 tensors** written (train 1875, val 210), 0 errors, and `manifest.csv` with 2085 rows — the dataset the training notebook consumes.

In [ ]:
main()

Aegis-Safe-Work | FallDetection ETL Stage 2
Started: 2026-06-29T15:01:37.881001+00:00

[TRAIN / FALL] 811 videos
  [100/811] OK: Fall_0303.npy | rate=1.2 t/s | ETA=9.5 min
  [200/811] OK: Fall_0867.npy | rate=1.2 t/s | ETA=8.2 min
  [300/811] OK: Fall_0866.npy | rate=1.3 t/s | ETA=6.7 min
  [400/811] OK: Fall_0298.npy | rate=1.3 t/s | ETA=5.4 min
  [500/811] OK: Fall_0283.npy | rate=1.3 t/s | ETA=4.0 min
  [600/811] OK: Fall_0219.npy | rate=1.3 t/s | ETA=2.7 min
  [700/811] OK: Fall_0246.npy | rate=1.3 t/s | ETA=1.4 min
  [800/811] OK: Fall_0168.npy | rate=1.3 t/s | ETA=0.1 min
  [811/811] OK: Fall_0274.npy | rate=1.3 t/s | ETA=0.0 min

[TRAIN / NORMAL] 1064 videos
  [100/1064] OK: Normal_0638.npy | rate=1.0 t/s | ETA=15.8 min
  [200/1064] OK: Normal_0483.npy | rate=1.1 t/s | ETA=13.7 min
  [300/1064] OK: Normal_0027.npy | rate=1.1 t/s | ETA=11.8 min
  [400/1064] OK: Normal_0238.npy | rate=1.1 t/s | ETA=10.1 min
  [500/1064] OK: Normal_1154.npy | rate=1.1 t/s | ETA=8.6 min
  [600/1064]